### Runnable

A Runnable is an object in LangChain that represents something that can receive an input and produce an output.

The basic pattern is:

- Input → Runnable → Output

Many LangChain components such as prompts, LLMs, parsers, and custom functions implement the Runnable interface.

A Runnable is an executable component in LangChain.

In [2]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-20b")

### 1. RunnableSequence

RunnableSequence is used when you want to execute multiple Runnables one after another.

User Input
    ->
Prompt
    ->
Model
    ->
Parser
    ->
Final Output

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

#prompt
prompt = PromptTemplate(
    template='Explain {topic} in simple terms and maximum 200 words.',
    input_variables=['topic']
)

# parser 
parser = StrOutputParser()

# sequence 
chain = RunnableSequence(prompt,model,parser)

result = chain.invoke({"topic":"Gen AI"})

print(result)

**Generative AI** is a type of computer program that learns from lots of examples—like text, images, or music—and then creates new content that looks or sounds similar. Think of it as a super‑creative robot that has read (or seen) a huge library and can write a story, paint a picture, compose a tune, or answer questions by mixing the patterns it learned.

How it works in a nutshell:
1. **Training** – The AI is fed millions of examples and learns the statistical patterns (e.g., which words often follow each other).
2. **Generation** – When you give it a prompt, it uses those patterns to predict the next piece of content, one bit at a time, until it finishes the output.

Examples: ChatGPT writes essays; DALL‑E draws images from text; music‑generating AIs compose new songs. The goal is to help humans produce creative work faster, explore ideas, or automate routine content creation.


### 2. Runnable Parallel

RunnableParallel runs multiple chains using the same input.

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

explain_prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms.Max 200 words "
)

example_prompt = PromptTemplate.from_template(
    "Give one simple example of {topic}.Max 200 words"
)

advantage_prompt = PromptTemplate.from_template(
    "Give three advantages of {topic}.Max 200 words"
)

parser = StrOutputParser()

parallel_chain = RunnableParallel(
    explanation=explain_prompt | model | parser,
    example=example_prompt | model | parser,
    advantages=advantage_prompt | model | parser
)

result = parallel_chain.invoke({
    "topic": "Machine Learning"
})

print("EXPLANATION:")
print(result["explanation"])

print("\nEXAMPLE:")
print(result["example"])

print("\nADVANTAGES:")
print(result["advantages"])

EXPLANATION:
**Machine Learning (ML) – a quick, everyday‑language explanation**

Imagine teaching a child to recognize cats. You show many pictures, some of cats and some of dogs, and you tell the child which is which. After looking at enough examples, the child starts spotting the common patterns—pointy ears, whiskers, a certain shape of face—and can guess whether a new picture is a cat or not.

Machine learning does the same thing, but with computers. Instead of a child, we give a computer a *large set of labeled data* (e.g., thousands of pictures of cats and dogs). The computer uses math to find patterns in the data, creating a tiny “recipe” (a model) that can predict labels for new, unseen examples. Once trained, the model can classify emails, recommend music, translate languages, drive cars, and more—all without being explicitly programmed for each task.

In short, ML is about letting computers learn from data so they can make decisions or predictions on their own.

EXAMPLE:
**Exa

### 3. RunnablePassthrough

RunnablePassthrough simply passes the original input forward.

This becomes especially useful with RunnableParallel.

In [8]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Answer this question: {question}"
)

chain = RunnableParallel(
    question=RunnablePassthrough(),  #the input itself is stored here in the question
    answer=prompt | model | StrOutputParser()
)

result = chain.invoke(
    "What is Machine Learning?"
)

print(result)

{'question': 'What is Machine Learning?', 'answer': '**Machine Learning (ML)** is a branch of artificial intelligence that focuses on building systems that can learn from data, identify patterns, and make decisions or predictions with minimal human intervention.  \n\nKey points:\n\n1. **Data‑driven learning** – Instead of programming explicit rules, ML models discover relationships in data through statistical and algorithmic techniques.\n2. **Adaptation** – Models improve over time as they process more data or as their environment changes.\n3. **Types of learning**  \n   - *Supervised*: Learn from labeled examples (e.g., image classification).  \n   - *Unsupervised*: Discover hidden structure in unlabeled data (e.g., clustering).  \n   - *Semi‑supervised*: Use a mix of labeled and unlabeled data.  \n   - *Reinforcement*: Learn by interacting with an environment and receiving rewards (e.g., game playing).\n4. **Applications** – From recommendation engines and natural language processing

### 4. Runnable Lambda

Runnable lambda allows you to use a normal python function as part of your GenAI chain 



In [10]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def prepare_topic(topic):
    return topic.strip().lower()

prepare = RunnableLambda(prepare_topic)

prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms.Max 50 words"
)

chain = (
    prepare
    | (lambda x: {"topic": x})     # strips out the spaces in the topic and sends that to the prompt
    | prompt
    | model
    | StrOutputParser()
)

result = chain.invoke(
    "   MACHINE LEARNING   "
)

print(result)

Machine learning is a way to teach computers to recognize patterns and make predictions. You give them lots of examples, and they learn rules from those examples, so they can guess outcomes or classify new data without being explicitly programmed for each case.


### 5. RunnableBranch 

RunnableBranch is basically:
 
if / elif / else

for Runnable chains.

In [11]:
from langchain_core.runnables import RunnableBranch
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

short_prompt = PromptTemplate.from_template(
    "Answer this question in one or two sentences:\n{question}"
)

long_prompt = PromptTemplate.from_template(
    "Give a detailed explanation of this question:\n{question} maximum of 200 words"
)

short_chain = short_prompt | model | StrOutputParser()

long_chain = long_prompt | model | StrOutputParser()

branch = RunnableBranch(
    (
        lambda x: len(x["question"]) < 50,
        short_chain              #executes only if the length of the question is less than 50
    ),
    long_chain
)

result = branch.invoke({
    "question": "What is Python?"
})

print(result)

Python is a high‑level, interpreted programming language known for its readability, dynamic typing, and extensive standard library. It is widely used for web development, data analysis, artificial intelligence, and scripting.
